# INSTALL DEPENDENCIES

In [ ]:
INITAL_SETUP = False

if INITAL_SETUP:
    %pip install fastkaggle
    %pip install kaggle
    %pip install dotenv
    %pip install ipdb


In [ ]:
import fastkaggle
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
import os
from fastai.tabular import *
from fastai.tabular.all import *
from fastai.vision.all import *

PATH_LOCAL_PATH_DATA_STORAGE = Path(r'D:\kaggle_data')
COMPETITION_NAME = '104-flowers-garden-of-eden'


# CHECK IF RUNNING ON KAGGLE OR ELSEWHERE

In [ ]:
'Kaggle' if fastkaggle.iskaggle else 'Not Kaggle'

# PREPARE KAGGLE DATA

In [ ]:
# if fastkaggle.iskaggle:
#     path = os.path.join(Path('../input'), COMPETITION_NAME) # generates path containing competition name
    
# else:
#     path_temp = Path(COMPETITION_NAME)
    
#     # check if data is missing from the desired location
#     path_local_competition_data = os.path.join(Path(PATH_LOCAL_PATH_DATA_STORAGE), COMPETITION_NAME) # generates path containing competition name
#     if not os.path.isdir(path_local_competition_data):
#         fastkaggle.setup_comp(COMPETITION_NAME) # download data to temp location
#         dest = shutil.move(path_temp, PATH_LOCAL_PATH_DATA_STORAGE) # move the data to the correct location
    
#     path = path_local_competition_data # update path to point to folder containing data

# print("Data located at: {}".format(str(path)))
    
path = os.path.join(Path(PATH_LOCAL_PATH_DATA_STORAGE), COMPETITION_NAME) # generates path containing competition name

FORMAT_SIZE = "jpeg-192x192"
path = os.path.join(path, 'versions', '1', FORMAT_SIZE) # generates path containing competition name

path

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("msheriey/104-flowers-garden-of-eden")

# print("Path to dataset files:", path)

# Check out the images

In [ ]:
# dls = ImageDataLoaders.from_name_func(
#     path,                                # path to images 
#     get_image_files(path),               # get the list of path for all images
#     valid_pct = 0.25,                    # percentages to use for validation
#     seed = 42,                           # random seed
#     label_func = parent_label,          # labelling for the images
#     item_tfms = Resize(224))             # transform the images to 224x224


dls = ImageDataLoaders.from_folder(
    path,                                # path to images 
    train='train',
    valid='val',
    blocks = (ImageBlock, CategoryBlock),       # specify the type of data
    seed = 42,                           # random seed
    #label_func = parent_label,          # labelling for the images
    get_y = parent_label,
    item_tfms = Resize(224))             # transform the images to 224x224

dls.show_batch()

In [ ]:
for vocab in dls.vocab:
    print(vocab)

# Prepare dataset

TBD if any work needs to be done here. Currently just a placeholder


# Vision Learner

In [ ]:
learn = vision_learner(dls, 
                       resnet18,
                       metrics=[accuracy, error_rate])

In [ ]:
# import torchvision
# torchvision.__version__
# torch.__version__
# import pytorch
# pytorch.__version__

In [ ]:
# import torch
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

# Fine Tune
We fine tune this model because we want to take advantage of the pretrained model. This will sever the head and retrain the last layer for our specific application. 

In [ ]:
learn.fine_tune(1)

# Quick check of the results

In [ ]:
learn.show_results()

# Prepare testing data

Load testing data....

In [ ]:
test_files = get_image_files(os.path.join(path,'test'))
print("Found {} test files.".format(len(test_files)))

In [ ]:
dls_test = learn.dls.test_dl(test_files)

# Inference

In [ ]:
preds, _, decoded = learn.get_preds(dl=dls_test, with_decoded=True)

Let's check the results

In [ ]:
preds, decoded
predicted_classes = [learn.dls.vocab[i] for i in preds.argmax(dim=1)]
len(predicted_classes)
print('Length of preds:', len(preds))
print('Length of decoded:', len(decoded))
print('Length of predicted_classes:', len(predicted_classes))


In [ ]:
predicted_classes[:10]


In [ ]:
#learn.show_results(dl=dls_test, max_n=5)
#interp = Interpretation.from_learner(learn, dl=dls_test)
#interp.plot_top_losses(9)
#learn.dls.show_results(dls_test.one_batch(), preds, max_n=9)

In [ ]:
predicted_classes[0]

In [94]:
#path_submission = os.path.join(path, 'versions', '1', FORMAT_SIZE) # generates path containing competition name
path_submission = os.path.join(path, '..', 'sample_submission.csv')

submission_csv = pd.read_csv(path_submission, header=0)
submission_csv.head(10)
#submission_csv.iloc[0]

,id,label
0,b48c962e0,0
1,a13d3dfa4,0
2,94269c190,0
3,bcb18c6e4,0
4,d15a4d94c,0
5,914b0e71b,0
6,065c9b5be,0
7,dd8b1aaca,0
8,8c39d1b41,0
9,e2fdce920,0


# Adding predictions to the submission CSV 

In [ ]:
dls_test.items

In [86]:
# Get the original filenames/paths
file_names = [f.name.replace('.jpeg', '') for f in dls_test.items]
file_names[:10]

['001e13533',
 '0021f0d33',
 '003882deb',
 '0039e54f0',
 '003b89961',
 '00450d304',
 '0045ce94c',
 '004b88e09',
 '0053fa6c4',
 '006afe0ef']

In [ ]:
# Combine them into a dictionary or list for easy viewing
results = list(zip(file_names, predicted_classes))
results[:10]
d_results = dict(results)
#d_results

In [101]:
print(submission_csv.loc[0, 'id'] )
print(submission_csv.loc[0, 'label'] )

b48c962e0
0


In [ ]:
submission_csv['label'] = submission_csv['label'].astype(str)

for index, row in submission_csv.iterrows():
    #print('index', index)
    #print('row id', row['id'])
    #print('row label', row['label'])
    matched_index = d_results.get(row['id'])
    #print('new label', matched_index)
    submission_csv.loc[index, 'label'] = matched_index
    #print('row label after', row['label'])
    # if index>10:
    #     break'

index 0
new label corn poppy
index 1
new label petunia
index 2
new label rose
index 3
new label wild geranium
index 4
new label common dandelion
index 5
new label sweet william
index 6
new label pink-yellow dahlia
index 7
new label watercress
index 8
new label cosmos
index 9
new label wild geranium
index 10
new label wild rose
index 11
new label wild pansy
index 12
new label yellow iris
index 13
new label colt's foot
index 14
new label toad lily
index 15
new label mexican petunia
index 16
new label common tulip
index 17
new label bougainvillea
index 18
new label iris
index 19
new label rose
index 20
new label anthurium
index 21
new label mallow
index 22
new label grape hyacinth
index 23
new label purple coneflower
index 24
new label daffodil
index 25
new label black-eyed susan
index 26
new label iris
index 27
new label wild rose
index 28
new label lenten rose
index 29
new label iris
index 30
new label buttercup
index 31
new label azalea
index 32
new label wild geranium
index 33
new lab

In [108]:
submission_csv.head(10)

,id,label
0,b48c962e0,corn poppy
1,a13d3dfa4,petunia
2,94269c190,rose
3,bcb18c6e4,wild geranium
4,d15a4d94c,common dandelion
5,914b0e71b,sweet william
6,065c9b5be,pink-yellow dahlia
7,dd8b1aaca,watercress
8,8c39d1b41,cosmos
9,e2fdce920,wild geranium


Preprocess testing data...

In [ ]:
preprocessed_test_df = preprocess(test_df)
preprocessed_test_df.head(5)

Check to see if any issues are in the testing data (e.g. n/a)

In [ ]:
preprocessed_test_df.isnull().any()

In [ ]:
raw_preds, _, decoded = learn.get_preds(dl=dls_test, with_decoded=True)

In [ ]:
raw_preds[:5], decoded[:5]

In [ ]:
dls_test.show_batch()

In [ ]:
df_submission = pd.DataFrame({'PassengerId': preprocessed_test_df.PassengerId
                              , 'Survived': decoded.flatten()})

In [ ]:
df_submission.head(5)


In [ ]:
df_submission.to_csv(os.path.join(path, 'submission.csv'), index=False)

# Test data prediction

In [ ]:
# tpreds=[]
# for i in range(len(preprocessed_test_df)):
#     _, _, probs = learn.predict(preprocessed_test_df.iloc[i])
#     tpreds+=[probs.numpy()[0]]